## 案例1：使用Pydantic定义args_schema工具输入

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field

from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv()

model = init_chat_model(
    model = "deepseek-v4-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL")
)


class WeatherSchema(BaseModel):
    city: str = Field(default="北京", description="城市名称")
    if_forecast: bool = Field(default=False, description="是否包含明日天气预报")


@tool("get_weather_and_forecast", description="查询当日天气，可以包含明日天气预报", args_schema=WeatherSchema)
def get_weather(city: str, if_forecast: bool):
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天也不错"
    return res


print(convert_to_openai_tool(get_weather))

model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage("今天杭州天气如何？明天呢？")]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)

final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日天气，可以包含明日天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明日天气预报', 'type': 'boolean'}}, 'type': 'object'}}}
================================ Human Message =================================

今天杭州天气如何？明天呢？
================================== Ai Message ==================================

好的，我来查一下杭州今天和明天的天气情况。
Tool Calls:
  get_weather_and_forecast (call_00_xpHTQ9SUxAALA0p0392v1386)
 Call ID: call_00_xpHTQ9SUxAALA0p0392v1386
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州 今天天气不错
明天也不错
================================== Ai Message ==================================

根据查询结果，为您总结如下：

### 🌤 杭州天气情况

| 日期 | 天气状况 |
|------|---------|
| **今天** | ✅ 天气不错 |
| **明天** | ✅ 天气也不错 |

总体来看，杭州今明两天天气都还不

## 案例2：使用docstring定义工具

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field

from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv()

model = init_chat_model(
    model = "deepseek-v4-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL")
)



## 案例3：多工具调用

In [5]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司，微软公司，谷歌公司）
        timeframe: 时间范围（today-今日，week-本周，month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格：{price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"

# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }

    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

# rprint(convert_to_openai_tool(search_news))

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)

message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

# 3.工具调用循环
while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)

    print("模型第一次响应是否包含工具调用:", response.tool_calls is not None)


    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    # 如果有调用工具，处理工具调用响应
    # 4.开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            #print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            #print("news_result", news_result)
            message_list.append(news_result)

# print("response", response)
print(response.content)

# for msg in message_list:
#     msg.pretty_print()

模型第一次响应是否包含工具调用: True
模型第一次响应是否包含工具调用: True
没有工具调用，直接返回答案
## 📊 苹果公司今日股价

**当前股价：185.20 美元**（今日数据）

---

## 📰 最新新闻动态

以下是苹果公司近期的几条重要新闻：

1. **🍎 苹果发布新款iPhone，股价上涨3%**
   - 新款iPhone的发布推动了股价上涨，市场反应积极。

2. **⚖️ 苹果与欧盟达成反垄断和解协议**
   - 苹果与欧盟在反垄断问题上达成了和解，这对公司的合规风险有所缓解。

3. **🏭 苹果将在印度扩大生产规模**
   - 苹果继续推进供应链多元化，计划在印度进一步扩大产能。

---

如需了解更详细的信息，比如本周或本月的股价走势，欢迎随时告诉我！😊
